# ICU Lactate Cohort

## Cohort Extraction

This notebook extracts the ICU lactate cohort from the MIMIC-III database.

The goal is to identify ICU stays with at least one lactate measurement within the first 24 hours after ICU admission. The resulting cohort will later be used to analyze the association between lactate levels and ICU mortality.

The extracted dataset includes:
- patient demographics (age, gender)
- ICU admission and discharge times
- mortality indicators
- the first recorded lactate measurement within 24 hours after ICU admission

The resulting dataset is exported as a CSV file for use in the main analysis notebook.

## Setup

In [ ]:
#! pip install pandas
#! pip install os
#! pip install pymysql
#! pip install dotenv

In [5]:
import pandas as pd
import os

# For MySQL/MariaDB (install with: pip install pymysql)
import pymysql

### Database connection

The notebook connects to the MIMIC-III relational database using credentials stored as environment variables.

To protect sensitive information, database credentials are not stored in the code. Instead, they are loaded from a local `.env` file.

This ensures that authentication details remain private and are not committed to version control.

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()


True

In [6]:
DB_HOST = os.environ['DB_HOST']
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_NAME = os.environ.get('DB_NAME', 'mimiciii')
DB_PORT = int(os.environ.get('DB_PORT', 3306))

con = pymysql.connect(
    host=DB_HOST,
    user=DB_USER,
    password=DB_PASSWORD,
    database=DB_NAME,
    port=DB_PORT
)

### Cohort definition

The cohort is defined using the following criteria:

1. ICU stays recorded in the `ICUSTAYS` table.
2. At least one lactate measurement recorded in the `LABEVENTS` table.
3. The lactate measurement must occur within the first 24 hours after ICU admission.
4. Only the **first lactate measurement** per ICU stay is selected.

The query also extracts demographic information and mortality indicators from the relevant tables.

In [7]:
COHORT_QUERY = """
WITH icu_ordered AS (
    SELECT
        i.subject_id, i.hadm_id, i.icustay_id, i.intime, i.outtime,
        a.hospital_expire_flag AS hospital_dead,
        ROW_NUMBER() OVER (PARTITION BY i.subject_id, i.hadm_id ORDER BY i.outtime DESC) AS rn_icu
    FROM ICUSTAYS i
    JOIN ADMISSIONS a ON i.hadm_id = a.hadm_id
),
lactate_first AS (
    SELECT io.subject_id, io.hadm_id, io.icustay_id, l.charttime as lactate_time, l.value as lactate_value,
           l.valuenum as lactate_value_num, l.valueuom as lactate_units, l.flag as lactate_flag,
           ROW_NUMBER() OVER (PARTITION BY io.icustay_id ORDER BY l.charttime ASC) AS rn
    FROM LABEVENTS l
    INNER JOIN icu_ordered io USING(subject_id, hadm_id)
    WHERE l.itemid = 50813 AND l.value IS NOT NULL
      AND l.charttime BETWEEN io.intime AND (io.intime + INTERVAL '24' HOUR)
),
icu_with_lactate AS (
    SELECT DISTINCT subject_id, hadm_id, icustay_id FROM lactate_first WHERE rn = 1
)
SELECT
    ROW_NUMBER() OVER (ORDER BY iu.subject_id, iu.hadm_id, iu.icustay_id) AS row_id,
    iu.subject_id, p.dob,
    YEAR(iu.intime) - YEAR(p.DOB) AS age_icu_entry,
    YEAR(iu.outtime) - YEAR(p.DOB) AS age_icu_exit,
    p.gender, iu.hadm_id, iu.icustay_id, iu.intime, iu.outtime, iu.hospital_dead,
    CASE WHEN iu.hospital_dead = 1 AND iu.rn_icu = 1 THEN 1 ELSE 0 END AS icu_dead,
    lf.lactate_time, lf.lactate_value, lf.lactate_value_num, lf.lactate_units, lf.lactate_flag
FROM icu_ordered iu
INNER JOIN icu_with_lactate iwl ON iwl.subject_id = iu.subject_id AND iwl.hadm_id = iu.hadm_id AND iwl.icustay_id = iu.icustay_id
LEFT JOIN lactate_first lf ON lf.subject_id = iu.subject_id AND lf.hadm_id = iu.hadm_id AND lf.icustay_id = iu.icustay_id AND lf.rn = 1
LEFT JOIN PATIENTS p ON p.subject_id = iu.subject_id
ORDER BY iu.subject_id, iu.hadm_id, iu.icustay_id;
"""

### Cohort extraction query

The SQL query uses several common table expressions (CTEs) to construct the cohort:

- `icu_ordered` identifies ICU stays and links them to hospital admissions.
- `lactate_first` identifies the first lactate measurement within the first 24 hours after ICU admission.
- `icu_with_lactate` selects ICU stays with a valid lactate measurement.

These steps ensure that each ICU stay is represented by a single lactate measurement taken early during the ICU stay.

## Extract and save to CSV

In [ ]:
cohort_df = pd.read_sql(COHORT_QUERY, con)
con.close()

OUTPUT_CSV = 'cohort_export.csv'
cohort_df.to_csv(OUTPUT_CSV, index=False)

print(f'Exported {len(cohort_df):,} rows to {OUTPUT_CSV}')

In [ ]:
cohort_df.info()
cohort_df.head()

### Exporting the cohort

The extracted cohort is exported as a CSV file (`cohort_export.csv`).

This file serves as the input dataset for the main statistical analysis notebook, where the association between lactate levels and ICU mortality will be evaluated.